# RB (native SX) + native-gate, on real IQM hardware

In [ ]:
import os
import json
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from qiskit import QuantumCircuit
from qiskit.circuit.library import SXGate
from qiskit.quantum_info import Operator
from qiskit_experiments.library import StandardRB, InterleavedRB

from iqm.qiskit_iqm import IQMProvider, transpile_to_IQM
from iqm.pulla.pulla import Pulla
from iqm.pulla.utils_qiskit import get_qiskit_compiler

import iqm_tools


## Connect to IQM hardware


In [ ]:
# IQM_TOKEN must be set as an environment variable before running this notebook.
# Never hardcode a real token in a notebook cell, especially in a public repo.
#   export IQM_TOKEN="<your-api-token-here>"   # set this in your shell, before launching Jupyter
if 'IQM_TOKEN' not in os.environ:
    raise RuntimeError(
        "Set the IQM_TOKEN environment variable before running this notebook "
        "(e.g. `export IQM_TOKEN=<your-api-token-here>` in your shell before launching Jupyter). "
        "Never hardcode a token in a notebook cell, especially in a public repo."
    )

iqm_server_url = 'https://resonance.iqm.tech/emerald'   # <- change to your target machine
qbt_name        = 'QB7'                                   # <- change to your target qubit

pulla    = Pulla(iqm_server_url)
backend  = IQMProvider(iqm_server_url).get_backend()
compiler = get_qiskit_compiler(pulla, backend)

print(f'Connected to {iqm_server_url}, targeting {qbt_name}')


## Inspect + freeze the native SX gate calibration

In [ ]:
# -- Inspect the native SX gate calibration (informational only) --------
_qc_ref   = QuantumCircuit(1); _qc_ref.sx(0)
_s_ref    = compiler.get_settings(circuits=[_qc_ref])
_impl     = _s_ref.gate_definitions.prx.default_implementation.value
_prx_node = _s_ref.gates.prx[_impl][qbt_name]

print(f'Native PRX implementation on {qbt_name}: {_impl}')
print(f'  duration    = {_prx_node.duration.value*1e9:.1f} ns')
print(f'  amplitude_i = {_prx_node.amplitude_i.value:.4f}')
print(f'  amplitude_q = {_prx_node.amplitude_q.value:.4f}')
print(f'  rz_before   = {_prx_node.rz_before.value:.6f} rad')
print(f'  rz_after    = {_prx_node.rz_after.value:.6f} rad')


In [ ]:
# -- Fixed native SX gate: freeze the calibrated parameters -----------------
# Captures every settable property of the native SX implementation ONCE
# (duration, amplitude_i, amplitude_q, rz_before, rz_after, full_width,
# center_offset) and defines get_fixed_native_settings(circuits), which forces
# every prx() call in a batch to use these EXACT frozen values instead of the
# compiler's live/default calibration lookup. This gives explicit, reproducible
# pulse parameters immune to any recalibration that might happen server-side
# during a long-running sweep, and an explicit baseline to compare against the
# live-calibration SRB/IRB results.
_qc_ref      = QuantumCircuit(1); _qc_ref.sx(0)
_s_ref       = compiler.get_settings(circuits=[_qc_ref])
_native_impl = _s_ref.gate_definitions.prx.default_implementation.value
_prx_node    = _s_ref.gates.prx[_native_impl][qbt_name]

fixed_native_params = {
    'implementation': _native_impl,
    'duration':       _prx_node.duration.value,
    'amplitude_i':    _prx_node.amplitude_i.value,
    'amplitude_q':    _prx_node.amplitude_q.value,
    'rz_before':      _prx_node.rz_before.value,
    'rz_after':       _prx_node.rz_after.value,
    'full_width':     _prx_node.full_width.value,
    'center_offset':  _prx_node.center_offset.value,
}

def get_fixed_native_settings(circuits):
    s    = compiler.get_settings(circuits=circuits)
    impl = fixed_native_params['implementation']
    node = s.gates.prx[impl][qbt_name]
    node.duration      = fixed_native_params['duration']
    node.amplitude_i   = fixed_native_params['amplitude_i']
    node.amplitude_q   = fixed_native_params['amplitude_q']
    node.rz_before     = fixed_native_params['rz_before']
    node.rz_after      = fixed_native_params['rz_after']
    node.full_width    = fixed_native_params['full_width']
    node.center_offset = fixed_native_params['center_offset']
    return s

print(f'Fixed native SX gate captured (impl={fixed_native_params["implementation"]}):')
print(f'  duration      = {fixed_native_params["duration"]*1e9:.1f} ns')
print(f'  amplitude_i   = {fixed_native_params["amplitude_i"]:.4f}')
print(f'  amplitude_q   = {fixed_native_params["amplitude_q"]:.4f}')
print(f'  rz_before     = {fixed_native_params["rz_before"]:.6f} rad')
print(f'  rz_after      = {fixed_native_params["rz_after"]:.6f} rad')


## RB parameters + circuit generation

In [ ]:
# physical_qubits: qiskit-experiments uses 0-indexed abstract qubits;
#   compiler.compile(..., components=[qbt_name]) maps qubit 0 -> qbt_name.
physical_qubits = (0,)

rb_lengths      = [1, 10, 30, 60, 100, 150, 200]   # Clifford sequence lengths
rb_num_samples  = 10                                # random sequences per length
rb_seed         = 42
n_shots         = 1000

print(f'physical_qubits = {physical_qubits}  ({qbt_name})')
print(f'Total Standard RB circuits    : {len(rb_lengths) * rb_num_samples}')
print(f'Total Interleaved RB circuits : {len(rb_lengths) * rb_num_samples * 2}  (ref + interleaved)')


In [ ]:
# -- Generate circuits (StandardRB/InterleavedRB for circuit gen only) ----
# optimize_single_qubits=False (NOT True): direct unitary-equivalence checking
# (next cell) confirmed transpile_to_IQM(optimize_single_qubits=True)'s
# single-qubit merging pass has a real bug -- for long enough Clifford chains
# it silently produces a circuit implementing the WRONG unitary (not just a
# less-optimal one). optimize_single_qubits=False uses a few more physical
# pulses per Clifford on average but is verified correct.
srb_exp   = StandardRB(physical_qubits, rb_lengths, num_samples=rb_num_samples, seed=rb_seed)
srb_circs = srb_exp.circuits()

srb_decomp = [c.decompose() for c in srb_circs]
srb_decomp = [transpile_to_IQM(c, backend, restrict_to_qubits=[qbt_name],
                                optimize_single_qubits=False, ignore_barriers=False)
              for c in srb_decomp]
srb_lengths_per_circ = [int(c.metadata['xval']) for c in srb_circs]

irb_exp   = InterleavedRB(SXGate(), physical_qubits, rb_lengths,
                           num_samples=rb_num_samples, seed=rb_seed)
irb_circs = irb_exp.circuits()

irb_decomp = [c.decompose() for c in irb_circs]
irb_decomp = [transpile_to_IQM(c, backend, restrict_to_qubits=[qbt_name],
                                optimize_single_qubits=False, ignore_barriers=False)
              for c in irb_decomp]
irb_lengths_per_circ = [int(c.metadata['xval']) for c in irb_circs]

def _is_interleaved(circ):
    m = circ.metadata
    if 'interleaved' in m:
        return bool(m['interleaved'])
    if 'circuit_type' in m:
        return 'interleaved' in str(m['circuit_type'])
    return False

irb_ref_mask = [not _is_interleaved(c) for c in irb_circs]
irb_int_mask = [    _is_interleaved(c) for c in irb_circs]

irb_ref_decomp           = [c for c, m in zip(irb_decomp, irb_ref_mask)           if m]
irb_int_decomp           = [c for c, m in zip(irb_decomp, irb_int_mask)           if m]
irb_ref_lengths_per_circ = [l for l, m in zip(irb_lengths_per_circ, irb_ref_mask) if m]
irb_int_lengths_per_circ = [l for l, m in zip(irb_lengths_per_circ, irb_int_mask) if m]

print(f'SRB: {len(srb_circs)} circuits')
print(f'IRB reference:   {len(irb_ref_decomp)} circuits')
print(f'IRB interleaved: {len(irb_int_decomp)} circuits')


In [ ]:
# -- Zero-cost unitary verification (no hardware calls) -------------------
# Directly tests whether decompose()+transpile_to_IQM(...) preserves the exact
# logical unitary intended by qiskit-experiments' RB circuits, given the known
# risk with the single-qubit optimization pass noted above.
def _check_unitary(before, after):
    b = before.remove_final_measurements(inplace=False)
    a = after.remove_final_measurements(inplace=False)
    return Operator(b).equiv(Operator(a))

def _check_set(name, pre_decomp, post_decomp, lengths_per_circ):
    seen = set()
    n_checked = n_ok = 0
    for pre, post, m in zip(pre_decomp, post_decomp, lengths_per_circ):
        if m in seen:
            continue
        seen.add(m)
        ok = _check_unitary(pre, post)
        n_checked += 1
        n_ok += int(ok)
        print(f'  {name}  m={m:3d}  {"OK" if ok else "*** MISMATCH ***"}')
    print(f'{name}: {n_ok}/{n_checked} lengths matched')

_srb_pre_decomp = [c.decompose() for c in srb_circs]
_check_set('SRB', _srb_pre_decomp, srb_decomp, srb_lengths_per_circ)

_irb_ref_circs = [c for c, m in zip(irb_circs, irb_ref_mask) if m]
_irb_int_circs = [c for c, m in zip(irb_circs, irb_int_mask) if m]
_check_set('IRB-ref', [c.decompose() for c in _irb_ref_circs], irb_ref_decomp, irb_ref_lengths_per_circ)
_check_set('IRB-int', [c.decompose() for c in _irb_int_circs], irb_int_decomp, irb_int_lengths_per_circ)


## Circuit execution + RB fitting helpers

In [ ]:
# -- Batched circuit execution via Pulla -----------------------------------
def _default_get_settings(circuits):
    return compiler.get_settings(circuits=circuits)


def run_rb_circuits(circs_decomposed, lengths_per_circ, n_shots, label='', batch_size=100,
                     get_settings_fn=_default_get_settings):
    """Runs a list of decomposed RB circuits through Pulla, batching up to
    `batch_size` circuits into one compile+submit+wait (one job instead of
    one per circuit). Returns P(|0>) per circuit, same order as input.

    get_settings_fn(circuits) -> settings: defaults to the compiler's live
    calibration; pass get_fixed_native_settings to force the frozen native
    SX gate parameters instead."""
    n = len(circs_decomposed)
    probs = [None] * n
    n_batches = (n + batch_size - 1) // batch_size
    print(f'Running {n} circuits in {n_batches} batch(es) of up to {batch_size}  '
          f'[{label}  shots={n_shots}] ...')

    for start in range(0, n, batch_size):
        chunk         = circs_decomposed[start:start + batch_size]
        chunk_lengths = lengths_per_circ[start:start + batch_size]

        s = get_settings_fn(chunk)
        s.set_shots(n_shots)

        jd, ctx = compiler.compile(circuits=chunk, components=[qbt_name], settings=s)
        job = pulla.submit_playlist(jd, context=ctx)
        job.wait_for_completion()

        if job.status != 'completed':
            print(f'  batch [{start+1}:{start+len(chunk)}/{n}] FAILED ({job.status})')
            for i in range(len(chunk)):
                probs[start + i] = float('nan')
            continue

        res = job.result(compiler)
        for i in range(len(chunk)):
            prob1_raw       = float(res.dataset['counter.result'].isel(circuit_index=i).values.flat[-1])
            probs[start + i] = 1.0 - prob1_raw

        print(f'  batch [{start+1:3d}:{start+len(chunk):3d}/{n}]  '
              f'm range {min(chunk_lengths)}-{max(chunk_lengths)}  done')

    return np.array(probs)

print('run_rb_circuits ready.')


In [ ]:
# -- RB fitting -------------------------------------------------------------
def _group_probs(lengths_per_circ, probs, lengths):
    """Group P(|0>) results by sequence length; return (mean, stderr) arrays."""
    by_len = {m: [] for m in lengths}
    for m, p in zip(lengths_per_circ, probs):
        if m in by_len:
            by_len[m].append(p)
    means   = np.array([np.nanmean(by_len[m]) for m in lengths])
    stderrs = np.array([np.nanstd(by_len[m]) / np.sqrt(len(by_len[m])) if len(by_len[m]) > 1 else 0.0
                        for m in lengths])
    return means, stderrs


def rb_fit(lengths, lengths_per_circ, probs):
    """Fit P(|0>) = A * alpha^m + B. Returns dict with alpha, alpha_err, A, B,
    epc, epc_err, probs_mean, probs_std."""
    means, stderrs = _group_probs(lengths_per_circ, probs, lengths)

    valid = ~np.isnan(means)
    if not valid.all():
        bad = [ell for ell, ok in zip(lengths, valid) if not ok]
        print(f'  rb_fit WARNING: no valid data for length(s) {bad} '
              f'(likely a failed hardware batch) -- fitting on remaining '
              f'{int(valid.sum())}/{len(lengths)} points')
    if valid.sum() < 3:
        raise RuntimeError(f'rb_fit: only {int(valid.sum())} valid length(s) -- '
                            'need >=3 to fit A*alpha^m+B. Re-run the failed batch.')

    fit_lengths = np.asarray(lengths)[valid]
    fit_means   = means[valid]
    fit_stderrs = stderrs[valid]

    def model(m, A, alpha, B):
        return A * alpha**m + B

    p0 = [0.5, 0.95, 0.5]
    popt, pcov = curve_fit(model, fit_lengths, fit_means,
                           p0=p0, sigma=fit_stderrs + 1e-6, absolute_sigma=True,
                           bounds=([0, 0, 0], [1, 1, 1]))
    perr = np.sqrt(np.diag(pcov))
    A, alpha, B = popt

    # A fit landing right on a bound (alpha~0 or ~1) is a degenerate/
    # non-converged result, not a real decay measurement.
    if alpha < 1e-3 or alpha > 1 - 1e-3:
        print(f'  rb_fit WARNING: alpha={alpha:.6f} is right at a fit bound -- '
              f'likely a degenerate fit; do not trust EPC/EPC_err.')
        print(f'    lengths      : {list(fit_lengths)}')
        print(f'    P(|0>) means : {[round(v, 4) for v in fit_means]}')

    d   = 2  # single qubit
    epc     = (d - 1) / d * (1 - alpha)
    epc_err = (d - 1) / d * perr[1]

    return dict(alpha=alpha, alpha_err=perr[1], A=A, B=B,
                epc=epc, epc_err=epc_err,
                probs_mean=means, probs_std=stderrs)


def irb_epc_gate(fit_ref, fit_int):
    """EPC_gate = (d-1)/d * (1 - alpha_int/alpha_ref) with error propagation."""
    d   = 2
    r   = fit_int['alpha'] / fit_ref['alpha']
    epc = (d - 1) / d * (1 - r)
    err = (d - 1) / d * r * np.sqrt(
        (fit_int['alpha_err'] / fit_int['alpha'])**2 +
        (fit_ref['alpha_err'] / fit_ref['alpha'])**2)
    return epc, err

print('rb_fit / irb_epc_gate ready.')


## Standard RB (live calibration)

In [ ]:
srb_probs = run_rb_circuits(srb_decomp, srb_lengths_per_circ, n_shots, label='SRB')


In [ ]:
srb_fit = rb_fit(rb_lengths, srb_lengths_per_circ, srb_probs)
print(f'SRB:  alpha={srb_fit["alpha"]:.5f}  EPC={srb_fit["epc"]:.4e} +/- {srb_fit["epc_err"]:.1e}  '
      f'Fidelity={1 - srb_fit["epc"]:.5f}')

m_fine = np.linspace(0, max(rb_lengths), 400)
fig, ax = plt.subplots(figsize=(8, 5))
for m, p in zip(srb_lengths_per_circ, srb_probs):
    ax.scatter(m, p, s=15, alpha=0.3, color='tab:blue')
ax.errorbar(rb_lengths, srb_fit['probs_mean'], yerr=srb_fit['probs_std'],
            fmt='o', color='tab:blue', capsize=4,
            label=f'alpha={srb_fit["alpha"]:.4f}  EPC={srb_fit["epc"]:.2e}')
ax.plot(m_fine, srb_fit['A'] * srb_fit['alpha']**m_fine + srb_fit['B'], '--', color='tab:blue')
ax.set_xlabel('Sequence length (Cliffords)')
ax.set_ylabel('Survival probability  P(|0>)')
ax.set_title(f'Standard RB  --  {qbt_name}')
ax.set_ylim(0.5, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


## Interleaved RB, SX (live calibration)

In [ ]:
irb_ref_probs = run_rb_circuits(irb_ref_decomp, irb_ref_lengths_per_circ,
                                n_shots, label='IRB-ref')

irb_int_probs = run_rb_circuits(irb_int_decomp, irb_int_lengths_per_circ,
                                n_shots, label='IRB-int (SX)')


In [ ]:
irb_ref_fit = rb_fit(rb_lengths, irb_ref_lengths_per_circ, irb_ref_probs)
irb_int_fit = rb_fit(rb_lengths, irb_int_lengths_per_circ, irb_int_probs)
epc_sx, epc_sx_err = irb_epc_gate(irb_ref_fit, irb_int_fit)
fidelity_sx = 1 - epc_sx

print(f'IRB reference:        alpha={irb_ref_fit["alpha"]:.5f}')
print(f'IRB interleaved (SX): alpha={irb_int_fit["alpha"]:.5f}')
print(f'EPC_SX = {epc_sx:.4e} +/- {epc_sx_err:.1e}   Fidelity_SX = {fidelity_sx:.5f}')

fig, ax = plt.subplots(figsize=(8, 5))
for probs_all, lpc, fit, label, color, fmt in [
    (irb_ref_probs, irb_ref_lengths_per_circ, irb_ref_fit, 'Reference', 'tab:blue', 'o'),
    (irb_int_probs, irb_int_lengths_per_circ, irb_int_fit, 'Interleaved (SX)', 'tab:red', 's'),
]:
    for m, p in zip(lpc, probs_all):
        ax.scatter(m, p, s=15, alpha=0.3, color=color)
    ax.errorbar(rb_lengths, fit['probs_mean'], yerr=fit['probs_std'],
                fmt=fmt, color=color, capsize=4, label=f'{label}  alpha={fit["alpha"]:.4f}')
    ax.plot(m_fine, fit['A'] * fit['alpha']**m_fine + fit['B'], '--', color=color)

ax.set_xlabel('Sequence length (Cliffords)')
ax.set_ylabel('Survival probability  P(|0>)')
ax.set_title(f'Interleaved RB (SX gate)  --  {qbt_name}\nEPC_SX = {epc_sx:.2e} +/- {epc_sx_err:.1e}')
ax.set_ylim(0.0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


## Standard + Interleaved RB, fixed native SX

In [ ]:
srb_probs_fixed = run_rb_circuits(srb_decomp, srb_lengths_per_circ, n_shots,
                                   label='SRB (fixed native SX)',
                                   get_settings_fn=get_fixed_native_settings)


In [ ]:
srb_fit_fixed = rb_fit(rb_lengths, srb_lengths_per_circ, srb_probs_fixed)
print(f'SRB (fixed native SX):  alpha={srb_fit_fixed["alpha"]:.5f}  '
      f'EPC={srb_fit_fixed["epc"]:.4e} +/- {srb_fit_fixed["epc_err"]:.1e}  '
      f'Fidelity={1 - srb_fit_fixed["epc"]:.5f}')

fig, ax = plt.subplots(figsize=(8, 5))
for m, p in zip(srb_lengths_per_circ, srb_probs_fixed):
    ax.scatter(m, p, s=15, alpha=0.3, color='tab:green')
ax.errorbar(rb_lengths, srb_fit_fixed['probs_mean'], yerr=srb_fit_fixed['probs_std'],
            fmt='o', color='tab:green', capsize=4,
            label=f'alpha={srb_fit_fixed["alpha"]:.4f}  EPC={srb_fit_fixed["epc"]:.2e}')
ax.plot(m_fine, srb_fit_fixed['A'] * srb_fit_fixed['alpha']**m_fine + srb_fit_fixed['B'], '--', color='tab:green')
ax.set_xlabel('Sequence length (Cliffords)')
ax.set_ylabel('Survival probability  P(|0>)')
ax.set_title(f'Standard RB -- fixed native SX  --  {qbt_name}')
ax.set_ylim(0.5, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
irb_ref_probs_fixed = run_rb_circuits(irb_ref_decomp, irb_ref_lengths_per_circ, n_shots,
                                       label='IRB-ref (fixed native SX)',
                                       get_settings_fn=get_fixed_native_settings)
irb_int_probs_fixed = run_rb_circuits(irb_int_decomp, irb_int_lengths_per_circ, n_shots,
                                       label='IRB-int (fixed native SX)',
                                       get_settings_fn=get_fixed_native_settings)


In [ ]:
irb_ref_fit_fixed = rb_fit(rb_lengths, irb_ref_lengths_per_circ, irb_ref_probs_fixed)
irb_int_fit_fixed = rb_fit(rb_lengths, irb_int_lengths_per_circ, irb_int_probs_fixed)
epc_sx_fixed, epc_sx_fixed_err = irb_epc_gate(irb_ref_fit_fixed, irb_int_fit_fixed)
fidelity_sx_fixed = 1 - epc_sx_fixed

print(f'IRB reference (fixed):        alpha={irb_ref_fit_fixed["alpha"]:.5f}')
print(f'IRB interleaved (fixed SX):   alpha={irb_int_fit_fixed["alpha"]:.5f}')
print(f'EPC_SX (fixed) = {epc_sx_fixed:.4e} +/- {epc_sx_fixed_err:.1e}   '
      f'Fidelity_SX (fixed) = {fidelity_sx_fixed:.5f}')

fig, ax = plt.subplots(figsize=(8, 5))
for probs_all, lpc, fit, label, color, fmt in [
    (irb_ref_probs_fixed, irb_ref_lengths_per_circ, irb_ref_fit_fixed, 'Reference', 'tab:blue', 'o'),
    (irb_int_probs_fixed, irb_int_lengths_per_circ, irb_int_fit_fixed, 'Interleaved (fixed SX)', 'tab:green', 's'),
]:
    for m, p in zip(lpc, probs_all):
        ax.scatter(m, p, s=15, alpha=0.3, color=color)
    ax.errorbar(rb_lengths, fit['probs_mean'], yerr=fit['probs_std'],
                fmt=fmt, color=color, capsize=4, label=f'{label}  alpha={fit["alpha"]:.4f}')
    ax.plot(m_fine, fit['A'] * fit['alpha']**m_fine + fit['B'], '--', color=color)

ax.set_xlabel('Sequence length (Cliffords)')
ax.set_ylabel('Survival probability  P(|0>)')
ax.set_title(f'Interleaved RB -- fixed native SX  --  {qbt_name}\nEPC_SX = {epc_sx_fixed:.2e} +/- {epc_sx_fixed_err:.1e}')
ax.set_ylim(0.0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print('== RB Summary ==')
print(f'Qubit: {qbt_name}')
print()
print('Live calibration (default compiler settings, per-call lookup):')
print(f'  Standard RB:      alpha={srb_fit["alpha"]:.5f}  EPC={srb_fit["epc"]:.4e} +/- {srb_fit["epc_err"]:.1e}  '
      f'Fidelity={1 - srb_fit["epc"]:.5f}')
print(f'  Interleaved (SX): EPC_SX={epc_sx:.4e} +/- {epc_sx_err:.1e}  Fidelity_SX={fidelity_sx:.5f}')
print()
print('Fixed native SX gate (frozen duration/amplitude_i/amplitude_q/rz_before/rz_after):')
print(f'  Standard RB:      alpha={srb_fit_fixed["alpha"]:.5f}  EPC={srb_fit_fixed["epc"]:.4e} +/- {srb_fit_fixed["epc_err"]:.1e}  '
      f'Fidelity={1 - srb_fit_fixed["epc"]:.5f}')
print(f'  Interleaved (SX): EPC_SX={epc_sx_fixed:.4e} +/- {epc_sx_fixed_err:.1e}  Fidelity_SX={fidelity_sx_fixed:.5f}')


## Save the result

In [ ]:
# Structured so the results can be re-plotted/re-analyzed later without
# re-running anything on hardware -- every raw probs array and every
# lengths_per_circ list needed to regroup/refit is included, not just the
# final fitted numbers. Each major section (live_calibration, fixed_native,
# noisy_native_sx_detuning_sweep) is optional -- if you didn't run that part
# of the notebook this time, it's silently skipped rather than crashing the save.
def _arr(a):
    """numpy array/scalar -> plain JSON-serializable list/float."""
    return [float(x) for x in a] if hasattr(a, '__iter__') else float(a)

def _fit_to_dict(fit):
    return {
        'alpha':      float(fit['alpha']),
        'alpha_err':  float(fit['alpha_err']),
        'A':          float(fit['A']),
        'B':          float(fit['B']),
        'epc':        float(fit['epc']),
        'epc_err':    float(fit['epc_err']),
        'probs_mean': _arr(fit['probs_mean']),
        'probs_std':  _arr(fit['probs_std']),
    }

def _try_section(name, builder):
    try:
        return builder()
    except NameError as e:
        print(f'  skipping "{name}" section (not run this session: {e})')
        return None

def _build_live_calibration():
    return {
        'srb': {'fit': _fit_to_dict(srb_fit), 'probs': _arr(srb_probs)},
        'irb': {
            'ref':        {'fit': _fit_to_dict(irb_ref_fit), 'probs': _arr(irb_ref_probs)},
            'int':        {'fit': _fit_to_dict(irb_int_fit), 'probs': _arr(irb_int_probs)},
            'epc_sx':     float(epc_sx),
            'epc_sx_err': float(epc_sx_err),
            'fidelity_sx': float(fidelity_sx),
        },
    }

def _build_fixed_native():
    return {
        'srb': {'fit': _fit_to_dict(srb_fit_fixed), 'probs': _arr(srb_probs_fixed)},
        'irb': {
            'ref':        {'fit': _fit_to_dict(irb_ref_fit_fixed), 'probs': _arr(irb_ref_probs_fixed)},
            'int':        {'fit': _fit_to_dict(irb_int_fit_fixed), 'probs': _arr(irb_int_probs_fixed)},
            'epc_sx':     float(epc_sx_fixed),
            'epc_sx_err': float(epc_sx_fixed_err),
            'fidelity_sx': float(fidelity_sx_fixed),
        },
    }


expr_params  = iqm_tools.get_qubit_params(iqm_server_url)
qbt_row      = expr_params[expr_params['qubit'] == qbt_name].iloc[0]
calib_set_id = expr_params.attrs.get('calibration_set_id', 'unknown')
machine      = iqm_server_url.rstrip('/').split('/')[-1]
ts           = datetime.now().strftime('%Y%m%d_%H%M%S')

record = {
    'timestamp':          datetime.now().isoformat(),
    'machine':            machine,
    'calibration_set_id': calib_set_id,
    'qubit':               qbt_name,
    'T1_us':              float(qbt_row['T1_us']),
    'T2_ramsey_us':       float(qbt_row['T2_ramsey_us']),

    'rb_params': {
        'physical_qubits': list(physical_qubits),
        'rb_lengths':      rb_lengths,
        'rb_num_samples':  rb_num_samples,
        'rb_seed':         rb_seed,
        'n_shots':         n_shots,
    },

    'native_gate_calibration': fixed_native_params,

    'lengths_per_circ': {
        'srb':     srb_lengths_per_circ,
        'irb_ref': irb_ref_lengths_per_circ,
        'irb_int': irb_int_lengths_per_circ,
    },
}

for _name, _builder in [
    ('live_calibration',               _build_live_calibration),
    ('fixed_native',                   _build_fixed_native)
]:
    _section = _try_section(_name, _builder)
    if _section is not None:
        record[_name] = _section

save_dir  = 'Results/RB'
os.makedirs(save_dir, exist_ok=True)
save_path = f'{save_dir}/rb_sx_{qbt_name}_{machine}_{ts}.json'
with open(save_path, 'w') as f:
    json.dump(record, f, indent=2)

print(f'Saved: {save_path}')
